In [13]:
from fastapi import FastAPI, Response
from fastapi.responses import StreamingResponse
import pickle
import io
import os
from PIL import Image
from io import BytesIO




In [14]:
# import the model
with open('..\models\mvp2best_logistic_model.pkl', 'rb') as file:
    model = pickle.load(file)
    

<>:2: SyntaxWarning: invalid escape sequence '\m'
<>:2: SyntaxWarning: invalid escape sequence '\m'
C:\Users\milo\AppData\Local\Temp\ipykernel_30764\4233852703.py:2: SyntaxWarning: invalid escape sequence '\m'
  with open('..\models\mvp2best_logistic_model.pkl', 'rb') as file:
c:\Users\milo\Desktop\publicprojectsMilo\.venv\Lib\site-packages\sklearn\base.py:442: InconsistentVersionWarning: Trying to unpickle estimator LogisticRegression from version 1.0.2 when using version 1.7.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
c:\Users\milo\Desktop\publicprojectsMilo\.venv\Lib\site-packages\sklearn\base.py:442: InconsistentVersionWarning: Trying to unpickle estimator IsotonicRegression from version 1.0.2 when using version 1.7.1. This might lead to breaking code or invalid results. Use at your own risk. For more info plea

In [16]:
# load test image
def load_image():
    img = Image.open("injury.png")
    buffer = BytesIO()
    img.save(buffer, format='PNG')
    buffer.seek(0)
    return buffer

injury_img = load_image()

In [6]:
app = FastAPI(
    title = 'Run injury prediction API',
    description = 'Preddicting injury risk based on GarminConnect running data'
)

In [19]:
@app.get("/")
async def root():
    return {"message": "test for injury_risk_pred is running!"}

In [18]:
@app.post("/predict_and_visualize/")
async def predict_and_visualize(X:int = 2):
    """
    perform spoof calculation and return an image
    """
    try:
        
        # output the number squared
        X = X * 2

        # handle the image
        img = injury_img

        

            
        return StreamingResponse(img, media_type="image/png")
    
    except Exception as e:
        return {"error": str(e)}, 500

    


### testing my py files!

In [15]:
from apicall_input import main_api_call 
from data_extraction_v2 import main_extract_transform
#from MVP2_data_extraction import main_extract_transform_memory

In [17]:
# call the two preprocessing functions
start_date, end_date, df_memory = main_api_call(email = 'milomoran123@gmail.com', password='1@Agarmin')
df = main_extract_transform(start_date, end_date, df_memory)

Garmin Connect API - Activity Downloader
Login successful!
Activity data for 'indoor_cardio|21-08-2025|20130260263.csv' loaded into DataFrame.
Activity data for 'running|20-08-2025|20116740584.csv' loaded into DataFrame.
Activity data for 'indoor_cardio|19-08-2025|20111594169.csv' loaded into DataFrame.
Activity data for 'indoor_cardio|18-08-2025|20096824015.csv' loaded into DataFrame.
Activity data for 'indoor_cardio|15-08-2025|20065742282.csv' loaded into DataFrame.
Activity data for 'indoor_cardio|14-08-2025|20059835179.csv' loaded into DataFrame.
Activity data for 'treadmill_running|13-08-2025|20044942352.csv' loaded into DataFrame.
Activity data for 'indoor_cardio|12-08-2025|20034672816.csv' loaded into DataFrame.
Activity data for 'walking|10-08-2025|20012275661.csv' loaded into DataFrame.
Activity data for 'running|10-08-2025|20010223403.csv' loaded into DataFrame.
Activity data for 'treadmill_running|08-08-2025|19990936196.csv' loaded into DataFrame.
Activity data for 'indoor_c

In [18]:
df.head()

,Date,Day1 total km,Day1 km z3+,Day1 km z5,Day2-3 nr.sessions,Day2-3 total km,Day2-3 km z3+,Day2-3 km z5,Day4-7 nr.sessions,Day4-7 total km,...,Week1 total km z3+,Week1 max km Z3+ one day,Week2 max km one day,Week2 total km z3+,Week2 max km Z3+ one day,5day/3W tot km ratio,5day/3W proportion km z3+,5day/3W nr. sessions ratio,5day/3W hours alternative training ratio,ACWR
0,2025-07-12,0.00,0.00,0.0,3.0,8.56,8.56,0.03,2.0,8.37,...,6.03,4.09,3.00,3.00,3.0,0.701,0.678,0.8,1.000,2.267
1,2025-07-13,5.05,4.05,0.0,3.0,8.56,8.56,0.03,1.0,3.31,...,8.00,6.00,5.09,5.09,3.0,0.804,0.541,0.8,0.792,2.284
2,2025-07-14,0.00,0.00,0.0,1.0,5.05,4.05,0.00,4.0,11.87,...,8.00,6.00,5.09,5.09,3.0,0.804,0.541,0.8,0.325,2.284
3,2025-07-15,0.00,0.00,0.0,1.0,5.05,4.05,0.00,3.0,8.56,...,8.08,4.06,5.09,8.32,5.0,1.000,0.435,1.0,0.351,2.461
4,2025-07-16,6.40,5.40,0.0,0.0,0.00,0.00,0.00,4.0,13.61,...,8.08,4.06,5.09,8.32,5.0,0.572,0.480,0.4,0.569,1.536


In [19]:
# Return statistics relating to user data
def normalize_user(row, mean_df, std_df):
    z = (row - mean_df) / std_df
    return z

# Calculate the means and standard deviations of all healthy events per athlete
def getMeanStd_user(data):
    # drop the date column while the normalisaition is going on
    data_no_date = data.drop(columns =['Date'], errors = 'ignore')
    mean = data_no_date.mean()
    std = data_no_date.std()
    std.replace(to_replace=0.0, value=0.01, inplace=True)
    return mean, std

user_test_means, user_test_std = getMeanStd_user(df.copy())
# Normalize the data
user_normalized = df.apply(lambda x: normalize_user(x, user_test_means,user_test_std), axis=1)
user_normalized = user_normalized.drop(columns=[ 'Date'], errors='ignore')



In [20]:
# make predictions
predictions = model.predict(user_normalized)
# make probability predictions
probs = model.predict_proba(user_normalized)[:, 1]

# create a df of predictions using the date column from dfday_user and the predictions
df['injury predictions'] = predictions
df['injury probabilities'] = probs
df[['Date','injury predictions','injury probabilities']].head(30)

# Save the predictions to a CSV file
df.to_csv('user activity_data_with_predictions.csv', index=False)



#plt.figure(figsize=(16,8))
#plt.plot(df['Date'],df['injury probabilities'])
# fix the date axis titles to use every 3rd date in the format mm/dd
# select the last 5 chars of the date string to get the mm/dd format
#plt.xticks(df['Date'][::5], rotation=20, ha='right')
# plot the probabilities over time with a rolling mean
plt.figure(figsize=(10,5))
plt.plot(df['Date'],df['injury probabilities'].rolling(window=5).mean())
plt.xticks(df['Date'][::5], rotation=45, ha='right')
plt.savefig('rolling_mean_plot.png')
# save and display the images
plt.show()

AttributeError: 'CalibratedClassifierCV' object has no attribute 'estimator'

In [21]:
print(model)
print(dir(model))

AttributeError: 'CalibratedClassifierCV' object has no attribute 'estimator'

In [22]:
# import the model
with open('..\models\mvp1_logistic_model.pkl', 'rb') as file:
    model_old = pickle.load(file)
    

<>:2: SyntaxWarning: invalid escape sequence '\m'
<>:2: SyntaxWarning: invalid escape sequence '\m'
C:\Users\milo\AppData\Local\Temp\ipykernel_30764\4136789945.py:2: SyntaxWarning: invalid escape sequence '\m'
  with open('..\models\mvp1_logistic_model.pkl', 'rb') as file:
c:\Users\milo\Desktop\publicprojectsMilo\.venv\Lib\site-packages\sklearn\base.py:442: InconsistentVersionWarning: Trying to unpickle estimator LogisticRegression from version 1.0.2 when using version 1.7.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
c:\Users\milo\Desktop\publicprojectsMilo\.venv\Lib\site-packages\sklearn\base.py:442: InconsistentVersionWarning: Trying to unpickle estimator IsotonicRegression from version 1.0.2 when using version 1.7.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please r

In [23]:
print(model_old)
print(dir(model_old))

AttributeError: 'CalibratedClassifierCV' object has no attribute 'estimator'